# 0. Load in the cleaned test and train data

In [127]:
import pandas as pd

test = pd.read_csv("/Users/nick/Library/CloudStorage/OneDrive-Personal/Programming projects/Team Union 2/Team-Union/Nick/churn_test_cleaned.csv")
test.info()
submission = test[["GuestID"]].copy()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1739 entries, 0 to 1738
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   GuestID         1739 non-null   int64 
 1   PromoCode       1739 non-null   object
 2   Region          1739 non-null   object
 3   AllInclusive    1739 non-null   int64 
 4   PackageType     1739 non-null   object
 5   VIP             1739 non-null   int64 
 6   RoomService     1739 non-null   int64 
 7   Dining          1739 non-null   int64 
 8   Retail          1739 non-null   int64 
 9   Spa             1739 non-null   int64 
 10  Entertainment   1739 non-null   int64 
 11  LoyaltyPoints   1739 non-null   int64 
 12  SurveyScore     1739 non-null   int64 
 13  DaysSinceEmail  1739 non-null   int64 
 14  BookingChannel  1739 non-null   object
 15  AgeGroup        1739 non-null   object
 16  ReferralSource  1739 non-null   object
 17  SharedRoom      1739 non-null   int64 
 18  RoomFloo

In [128]:
test.isna().sum()

GuestID           0
PromoCode         0
Region            0
AllInclusive      0
PackageType       0
VIP               0
RoomService       0
Dining            0
Retail            0
Spa               0
Entertainment     0
LoyaltyPoints     0
SurveyScore       0
DaysSinceEmail    0
BookingChannel    0
AgeGroup          0
ReferralSource    0
SharedRoom        0
RoomFloor         0
RoomNumber        0
RoomSide          0
dtype: int64

In [129]:
train = pd.read_csv("/Users/nick/Library/CloudStorage/OneDrive-Personal/Programming projects/Team Union 2/Team-Union/Nick/churn_train_cleaned.csv")
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6954 entries, 0 to 6953
Data columns (total 22 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   GuestID         6954 non-null   int64 
 1   PromoCode       6954 non-null   object
 2   Region          6954 non-null   object
 3   AllInclusive    6954 non-null   int64 
 4   PackageType     6954 non-null   object
 5   VIP             6954 non-null   int64 
 6   RoomService     6954 non-null   int64 
 7   Dining          6954 non-null   int64 
 8   Retail          6954 non-null   int64 
 9   Spa             6954 non-null   int64 
 10  Entertainment   6954 non-null   int64 
 11  LoyaltyPoints   6954 non-null   int64 
 12  SurveyScore     6954 non-null   int64 
 13  DaysSinceEmail  6954 non-null   int64 
 14  BookingChannel  6954 non-null   object
 15  AgeGroup        6954 non-null   object
 16  ReferralSource  6954 non-null   object
 17  Churned         6954 non-null   int64 
 18  SharedRo

In [130]:
train.isna().sum()

GuestID           0
PromoCode         0
Region            0
AllInclusive      0
PackageType       0
VIP               0
RoomService       0
Dining            0
Retail            0
Spa               0
Entertainment     0
LoyaltyPoints     0
SurveyScore       0
DaysSinceEmail    0
BookingChannel    0
AgeGroup          0
ReferralSource    0
Churned           0
SharedRoom        0
RoomFloor         0
RoomNumber        0
RoomSide          0
dtype: int64

In [131]:
#Create X and y variables for modeling

from sklearn.model_selection import train_test_split

X = train.drop('Churned', axis=1)
y = train['Churned']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 1. Train the model - This is where your model goes

In [132]:
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import roc_auc_score

# CatBoost handles categoricals natively — just tell it which columns are categorical.
# GuestID is an identifier, drop it from features.
drop_cols = ["GuestID", "Churned"]
X = train.drop(columns=drop_cols)
y = train["Churned"]

cat_features = [c for c in X.columns if X[c].dtype == "object"]
# CatBoost requires categorical columns to have no NaN — fill with a sentinel string.
X[cat_features] = X[cat_features].fillna("missing")

#from sklearn.model_selection import train_test_split
#X_train_cb, X_test_cb, y_train_cb, y_test_cb = train_test_split(
#    X, y, test_size=0.2, random_state=42, stratify=y
#)

neg, pos = (y_train == 0).sum(), (y_train == 1).sum()

cat = CatBoostClassifier(
    iterations=1000,
    depth=4,
    learning_rate=0.2255611979301099,
    l2_leaf_reg=0.2939990463887897,
    bagging_temperature=0.13866256060046356,
    random_strength=0.02945655242478888,
    border_count=212,
    min_data_in_leaf=52,
    scale_pos_weight=neg / pos,
    eval_metric="AUC",
    early_stopping_rounds=50,
    random_seed=42,
    verbose=False,
)

train_pool = Pool(X_train, y_train, cat_features=cat_features)
test_pool = Pool(X_test, y_test, cat_features=cat_features)

cat.fit(train_pool, eval_set=test_pool)

probs = cat.predict(X_test)
print(f"Test ROC-AUC:        {roc_auc_score(y_test, probs):.3f}")
print(f"Best iteration:      {cat.get_best_iteration()}")

#use scitkit-learn classification report to get precision, recall, f1-score, and support for the test set predictions
from sklearn.metrics import classification_report
y_pred = cat.predict(X_test)
print(classification_report(y_test, y_pred))

# Feature importance
import pandas as pd
fi = pd.DataFrame({
    "feature": X_train.columns,
    "importance": cat.get_feature_importance(train_pool),
}).sort_values("importance", ascending=False)
print(fi.to_string(index=False))

Test ROC-AUC:        0.818
Best iteration:      187
              precision    recall  f1-score   support

           0       0.83      0.81      0.82       718
           1       0.81      0.82      0.81       673

    accuracy                           0.82      1391
   macro avg       0.82      0.82      0.82      1391
weighted avg       0.82      0.82      0.82      1391

       feature  importance
           Spa   13.557355
 Entertainment   11.990162
        Region    9.443867
    RoomNumber    8.221188
  AllInclusive    8.178469
ReferralSource    7.315000
        Dining    6.482694
   RoomService    6.277061
     PromoCode    5.081155
     RoomFloor    4.390971
        Retail    3.476362
DaysSinceEmail    3.141167
       GuestID    2.943781
      RoomSide    2.772678
 LoyaltyPoints    2.524477
   PackageType    1.564021
      AgeGroup    1.244550
   SurveyScore    0.805052
           VIP    0.396936
BookingChannel    0.173809
    SharedRoom    0.019245


# 2. Test - This will produce the test data

In [133]:
test.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1739 entries, 0 to 1738
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   GuestID         1739 non-null   int64 
 1   PromoCode       1739 non-null   object
 2   Region          1739 non-null   object
 3   AllInclusive    1739 non-null   int64 
 4   PackageType     1739 non-null   object
 5   VIP             1739 non-null   int64 
 6   RoomService     1739 non-null   int64 
 7   Dining          1739 non-null   int64 
 8   Retail          1739 non-null   int64 
 9   Spa             1739 non-null   int64 
 10  Entertainment   1739 non-null   int64 
 11  LoyaltyPoints   1739 non-null   int64 
 12  SurveyScore     1739 non-null   int64 
 13  DaysSinceEmail  1739 non-null   int64 
 14  BookingChannel  1739 non-null   object
 15  AgeGroup        1739 non-null   object
 16  ReferralSource  1739 non-null   object
 17  SharedRoom      1739 non-null   int64 
 18  RoomFloo

In [134]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6954 entries, 0 to 6953
Data columns (total 20 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   PromoCode       6954 non-null   object
 1   Region          6954 non-null   object
 2   AllInclusive    6954 non-null   int64 
 3   PackageType     6954 non-null   object
 4   VIP             6954 non-null   int64 
 5   RoomService     6954 non-null   int64 
 6   Dining          6954 non-null   int64 
 7   Retail          6954 non-null   int64 
 8   Spa             6954 non-null   int64 
 9   Entertainment   6954 non-null   int64 
 10  LoyaltyPoints   6954 non-null   int64 
 11  SurveyScore     6954 non-null   int64 
 12  DaysSinceEmail  6954 non-null   int64 
 13  BookingChannel  6954 non-null   object
 14  AgeGroup        6954 non-null   object
 15  ReferralSource  6954 non-null   object
 16  SharedRoom      6954 non-null   int64 
 17  RoomFloor       6954 non-null   object
 18  RoomNumb

In [135]:
#run the test data against the model to get the probabilities for the submission file

#drop GueestID Column from the test data
test = test.drop(columns=["GuestID"])
probs = cat.predict(test)

#probs = xgb.predict(test)

#display the probs
print(probs)

#remove all columns except for the GuestID column and then add the probabilities as a new column called "Churned"
#submission = train[["GuestID"]].copy()
submission["Churned"] = probs

submission.info()

#output the csv file with the predictions
submission.to_csv("submission.csv", index=False)

CatBoostError: Invalid cat_features[7] = 20 value: index must be < 20.